In [10]:
import numpy as np
import pandas as pd
import sqlite3
import scipy.stats as stats

# Set random seed so data is reproducible
np.random.seed(32);

num_drivers = 1000;

# Driver IDs
driver_ids = [f"DRV_{i:04d}" for i in range(1, num_drivers + 1)];

# Assign rideshare drivers between full-time (20%) and part-time (80%)
is_full_time = np.random.choice([1, 0], size=num_drivers, p=[0.20, 0.80]);

# Initialize data groups
annual_miles = [];
hard_brakes = [];
speeding_min = [];
night_driving_pct = [];

# Generate telematics data based on driver classification
count = 0;
for status in is_full_time:
    if status == 1: # Full-time Driver Profile
        miles = np.random.gamma(shape=12.25, scale=2857); # higher mileage
        brakes = np.random.gamma(shape=2.25, scale=0.11); # right skewed data --> gamma distribution
        speeding = np.random.gamma(shape=1.2, scale=5.0);
        night_driving = np.random.beta(a=2.0, b=6.0); # 0 to 1 boundary for percentages --> beta distribution
    else: # Part-time Drivers Profile
        miles = np.random.exponential(scale=7500); # more data points clustered near 0
        brakes = np.random.gamma(shape=1.40, scale=0.25); # greater variance
        speeding = np.random.gamma(shape=0.85, scale=8.23);
        night_driving = np.random.beta(a=0.5, b=0.27); # dip in  middle of distribution

    # Add to lists with rounding to clean data
    annual_miles.append(int(np.clip(miles, 20000, 110000)));
    hard_brakes.append(round(brakes, 1));
    speeding_min.append(round(speeding, 1));
    night_driving_pct.append(round(night_driving, 3));

print(night_driving_pct);

df = pd.DataFrame({
    'Driver_ID': driver_ids,
    'Is_Full_Time': is_full_time,
    'Annual_Miles': annual_miles,
    'Hard_Brakes_Per_100m': hard_brakes,
    'Speeding_Min_Per_100m': speeding_min,
    'Night_Driving_Pct': night_driving_pct
});

# Adding Human Error: introduce missing values to simulate real world data
df.loc[np.random.choice(df.index, size=15, replace=False), 'Hard_Brakes_Per_100m'] = np.nan;
df.loc[np.random.choice(df.index, size=10, replace=False), 'Speeding_Min_Per_100m'] = np.nan;

# Connect to SQLite database
conn = sqlite3.connect('rideshare_telematics.db');

# Save pandas dataframe into a SQL table
df.to_sql('raw_telematics', conn, if_exists='replace', index=False);

conn.close();

print(df.groupby('Is_Full_Time')[['Annual_Miles', 'Hard_Brakes_Per_100m', 'Night_Driving_Pct']].mean());

[0.951, 0.999, 0.995, 0.76, 0.976, 0.35, 0.362, 0.492, 0.999, 0.583, 0.211, 1.0, 1.0, 0.932, 0.517, 0.509, 0.09, 0.282, 0.023, 0.972, 0.02, 0.246, 1.0, 0.966, 0.996, 0.64, 0.083, 0.153, 0.095, 0.078, 0.315, 0.898, 0.037, 0.61, 0.102, 0.62, 0.98, 0.235, 0.999, 0.755, 0.138, 1.0, 1.0, 0.973, 0.096, 0.431, 0.032, 0.445, 0.09, 0.202, 0.759, 0.766, 0.999, 0.057, 0.451, 0.998, 0.852, 0.887, 0.025, 0.86, 0.426, 0.179, 0.502, 0.067, 0.354, 0.58, 0.988, 0.3, 0.132, 0.941, 0.999, 0.533, 0.484, 1.0, 0.998, 0.151, 0.651, 0.148, 0.265, 0.403, 0.048, 1.0, 0.385, 0.828, 0.173, 0.515, 0.504, 0.915, 0.076, 0.084, 0.95, 1.0, 0.959, 0.049, 0.932, 0.532, 0.367, 0.775, 0.997, 1.0, 0.771, 0.836, 0.11, 1.0, 0.223, 0.069, 0.33, 0.078, 0.199, 0.145, 1.0, 0.846, 0.998, 0.962, 1.0, 0.288, 0.036, 0.914, 0.001, 0.379, 0.931, 0.031, 0.442, 0.992, 0.799, 0.969, 0.396, 0.715, 0.183, 0.296, 0.618, 0.911, 1.0, 0.282, 0.972, 0.001, 0.894, 0.254, 0.259, 0.993, 0.839, 0.227, 0.16, 0.998, 0.784, 0.001, 0.346, 0.773, 0.302,